<a href="https://colab.research.google.com/github/mcfvdesigns/Site-Turnover-Defects/blob/main/Baseline_inference.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [24]:
!pip install -q ultralytics pyyaml

In [15]:
import os, hashlib, zipfile, urllib.request, yaml
from pathlib import Path

DATA_URL = "https://github.com/mcfvdesigns/Site-Turnover-Defects/releases/download/v4.0/Site.Turnover.Defects.v4i.yolov11.1.zip"
SHA256 = "443eced074cc61c29b5fd3e0b8a0ce3bb5d6de410312312acc5ce936a10a84bc"
ROOT = Path("/content/dataset")
ZIP_PATH = Path("/content/dataset.zip")

def sha256_file(path, chunk=1 << 20):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(chunk), b""):
            h.update(block)
    return h.hexdigest()

def find_data_yaml(root: Path) -> Path:
    hits = list(root.rglob("data.yaml"))
    if not hits:
        raise FileNotFoundError(f"No data.yaml under {root}")
    return hits[0]

already_ready = (ROOT / "data.yaml").exists() or any(ROOT.glob("*/data.yaml"))
if not already_ready:
    ROOT.mkdir(parents=True, exist_ok=True)
    if ZIP_PATH.exists():
        ZIP_PATH.unlink()
    urllib.request.urlretrieve(DATA_URL, ZIP_PATH)
    digest = sha256_file(ZIP_PATH)
    if digest != SHA256:
        ZIP_PATH.unlink(missing_ok=True)
        raise AssertionError(f"Checksum mismatch: {digest}")
    with zipfile.ZipFile(ZIP_PATH) as z:
        z.extractall(ROOT)

cfg_path = find_data_yaml(ROOT)
dataset_root = cfg_path.parent
cfg = yaml.safe_load(cfg_path.read_text())
cfg["path"] = str(dataset_root)
cfg["train"] = "train/images"
cfg["val"] = "valid/images"
if (dataset_root / "test" / "images").is_dir():
    cfg["test"] = "test/images"
else:
    cfg.pop("test", None)
cfg_path.write_text(yaml.safe_dump(cfg, sort_keys=False))
print(cfg)

{'train': 'train/images', 'val': 'valid/images', 'test': 'test/images', 'nc': 5, 'names': ['cracked jamb', 'cracked tile', 'moisture stain', 'paint defect', 'uneven joint'], 'roboflow': {'workspace': 'skipper-_sky', 'project': 'site-turnover-defects', 'version': 4, 'license': 'CC BY 4.0', 'url': 'https://universe.roboflow.com/skipper-_sky/site-turnover-defects/dataset/4'}, 'path': '/content/dataset'}


In [26]:
from ultralytics import YOLO

# 1. Load an un-finetuned, pre-trained base model
baseline_model = YOLO("yolo11n.pt")

# 2. Run baseline inference on test images
results = baseline_model.predict(
    source=str(dataset_root / "test/images"),
    save=True,
    conf=0.25
)

# 3. Evaluate baseline zero-shot performance on the validation split
metrics = baseline_model.val(data=str(cfg_path))
print("Baseline mAP50-95:", metrics.box.map)


image 1/5 /content/dataset/test/images/39C7A391-E9C7-4E96-9517-6D375401DEB8_heic.rf.5db3a079d7a90bf32e4eb67064162c4b.jpg: 640x640 (no detections), 8.1ms
image 2/5 /content/dataset/test/images/3B0369CD-D1EE-472C-9F32-4B2AF3395297_heic.rf.a5cc760ef037a0c61012c6bef46a2a9a.jpg: 640x640 (no detections), 13.6ms
image 3/5 /content/dataset/test/images/3C0BF6B7-BD58-4F89-9809-2541F0FF267A_heic.rf.b349f558e2e2c53882e350e3996622b9.jpg: 640x640 (no detections), 11.5ms
image 4/5 /content/dataset/test/images/5D20FB4A-8995-4278-8FB0-38F0F16F3222_heic.rf.90e31dfec36c4df4657b053fab35876d.jpg: 640x640 1 motorcycle, 1 bed, 8.4ms
image 5/5 /content/dataset/test/images/6CCD60CE-BE5E-41A1-8675-DFD83020914E_heic.rf.e1e4e1cb8897a6f59a1ae0cb7e67ae04.jpg: 640x640 1 person, 9.4ms
Speed: 1.4ms preprocess, 10.2ms inference, 0.8ms postprocess per image at shape (1, 3, 640, 640)
Results saved to /content/runs/detect/predict-6
Ultralytics 8.4.163 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
YOLO11

In [27]:
import os, hashlib, zipfile, urllib.request, yaml
from pathlib import Path

DATA_URL = "https://github.com/mcfvdesigns/Site-Turnover-Defects/releases/download/v4.0/Site.Turnover.Defects.v4i.yolov11.1.zip"
SHA256 = "443eced074cc61c29b5fd3e0b8a0ce3bb5d6de410312312acc5ce936a10a84bc"
ROOT = Path("/content/dataset")
ZIP_PATH = Path("/content/dataset.zip")

def sha256_file(path, chunk=1 << 20):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(chunk), b""):
            h.update(block)
    return h.hexdigest()

def find_data_yaml(root: Path) -> Path:
    hits = list(root.rglob("data.yaml"))
    if not hits:
        raise FileNotFoundError(f"No data.yaml under {root}")
    return hits[0]

already_ready = (ROOT / "data.yaml").exists() or any(ROOT.glob("*/data.yaml"))
if not already_ready:
    ROOT.mkdir(parents=True, exist_ok=True)
    if ZIP_PATH.exists():
        ZIP_PATH.unlink()
    urllib.request.urlretrieve(DATA_URL, ZIP_PATH)
    digest = sha256_file(ZIP_PATH)
    if digest != SHA256:
        ZIP_PATH.unlink(missing_ok=True)
        raise AssertionError(f"Checksum mismatch: {digest}")
    with zipfile.ZipFile(ZIP_PATH) as z:
        z.extractall(ROOT)

cfg_path = find_data_yaml(ROOT)
dataset_root = cfg_path.parent
cfg = yaml.safe_load(cfg_path.read_text())
cfg["path"] = str(dataset_root)
cfg["train"] = "train/images"
cfg["val"] = "valid/images"
if (dataset_root / "test" / "images").is_dir():
    cfg["test"] = "test/images"
else:
    cfg.pop("test", None)
cfg_path.write_text(yaml.safe_dump(cfg, sort_keys=False))
print(cfg)


{'train': 'train/images', 'val': 'valid/images', 'test': 'test/images', 'nc': 5, 'names': ['cracked jamb', 'cracked tile', 'moisture stain', 'paint defect', 'uneven joint'], 'roboflow': {'workspace': 'skipper-_sky', 'project': 'site-turnover-defects', 'version': 4, 'license': 'CC BY 4.0', 'url': 'https://universe.roboflow.com/skipper-_sky/site-turnover-defects/dataset/4'}, 'path': '/content/dataset'}


In [28]:
from ultralytics import YOLO

# 1. Initialize a fresh YOLO11 model
model = YOLO("yolo11n.pt")

# 2. Train the model on your dataset using the config path defined in Cell 2
train_results = model.train(
    data=str(cfg_path),
    epochs=50,
    imgsz=640,
    plots=True
)

# 3. Evaluate the newly fine-tuned model on the validation set
val_metrics = model.val(data=str(cfg_path))
print("Fine-tuned mAP50-95:", val_metrics.box.map)

Ultralytics 8.4.163 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/dataset/data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train-6, nbs=64, nms=None, opset=None, optimize=Fa

In [29]:
# Run inference on a new batch of test images
inference_results = model.predict(
    source=str(dataset_root / "test/images"),
    save=True,
    conf=0.25
)

print("Inference completed and results saved.")


image 1/5 /content/dataset/test/images/39C7A391-E9C7-4E96-9517-6D375401DEB8_heic.rf.5db3a079d7a90bf32e4eb67064162c4b.jpg: 640x640 (no detections), 7.9ms
image 2/5 /content/dataset/test/images/3B0369CD-D1EE-472C-9F32-4B2AF3395297_heic.rf.a5cc760ef037a0c61012c6bef46a2a9a.jpg: 640x640 (no detections), 8.6ms
image 3/5 /content/dataset/test/images/3C0BF6B7-BD58-4F89-9809-2541F0FF267A_heic.rf.b349f558e2e2c53882e350e3996622b9.jpg: 640x640 (no detections), 8.2ms
image 4/5 /content/dataset/test/images/5D20FB4A-8995-4278-8FB0-38F0F16F3222_heic.rf.90e31dfec36c4df4657b053fab35876d.jpg: 640x640 (no detections), 8.4ms
image 5/5 /content/dataset/test/images/6CCD60CE-BE5E-41A1-8675-DFD83020914E_heic.rf.e1e4e1cb8897a6f59a1ae0cb7e67ae04.jpg: 640x640 (no detections), 8.1ms
Speed: 1.4ms preprocess, 8.2ms inference, 0.5ms postprocess per image at shape (1, 3, 640, 640)
Results saved to /content/runs/detect/predict-7
Inference completed and results saved.


In [23]:
model.train(data=str(cfg_path), epochs=100, imgsz=640, freeze=10)

Ultralytics 8.4.163 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/dataset/data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=10, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=/content/runs/detect/train-4/weights/best.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train-5, nbs=64, 

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([2, 4])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7b1c58d1d0f0>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.04804

In [14]:
import os
import shutil
from pathlib import Path
from ultralytics import YOLO
from google.colab import files

# 1. Setup local folders
os.makedirs("/content/results/curves", exist_ok=True)
os.makedirs("/content/results/evidence", exist_ok=True)

# 2. Copy Ultralytics training curves (if available from runs)
runs_dir = Path("/content/runs/detect")
if runs_dir.exists():
    for plot in ["results.png", "PR_curve.png", "F1_curve.png", "P_curve.png", "confusion_matrix.png"]:
        for file_path in runs_dir.rglob(plot):
            shutil.copy(file_path, f"/content/results/curves/{plot}")

# 3. Load model and run inference on validation & test sets
best_pt_path = Path("/content/best.pt") # Directly use the specified path

if not best_pt_path.exists():
    raise FileNotFoundError(f"Could not find model weights at {best_pt_path}. Please ensure a training run has completed successfully and saved the model to this location.")

model = YOLO(str(best_pt_path))

# Save validation predictions
model.val(data="/content/dataset/data.yaml", save=True, project="/content/results/evidence", name="val_predictions")

# Save test set predictions
model.predict(source="/content/dataset/test/images", save=True, project="/content/results/evidence", name="test_predictions")

# 4. Zip the results folder for easy upload
shutil.make_archive("/content/results", 'zip', "/content/results")

# Download results.zip directly to your computer
files.download("/content/results.zip")

Ultralytics 8.4.163 🚀 Python-3.13.15 torch-2.11.0+cpu CPU (Intel Xeon CPU @ 2.20GHz)
YOLO11n summary (fused): 100 layers, 2,583,127 parameters, 0 gradients, 6.4 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 632.2±349.4 MB/s, size: 32.9 KB)
val: Scanning /content/dataset/valid/labels... 3 images, 1 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 3/3 67.3it/s 0.0s
val: New cache created: /content/dataset/valid/labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 1/1 1.1it/s 0.9s
                   all          3          3    0.00805          1     0.0684      0.035
        moisture stain          1          1    0.00893          1     0.0255     0.0128
          uneven joint          2          2    0.00717          1      0.111     0.0572
Speed: 6.2ms preprocess, 235.3ms inference, 0.0ms loss, 24.1ms postprocess per image
Results saved to /content/results/evidence/val_predictions-3

image 1/5 /content/dataset/test

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>